In [1]:
from pathlib import Path

import pandas as pd
import numpy as np


# ============================================
# PROJECT ROOT
# ============================================

ROOT = Path(
    r"D:\My_Project\Analyse\capston Project_2-Global Supply Chain Risk & Logistics"
)


# ============================================
# INPUT PATH
# ============================================

INPUT_PATH = (
    ROOT
    / "data"
    / "interim"
    / "cleaned_data.csv"
)


# ============================================
# OUTPUT PATH
# ============================================

OUTPUT_PATH = (
    ROOT
    / "data"
    / "interim"
    / "feature_engineered_data.csv"
)


# ============================================
# LOAD CLEANED DATA
# ============================================

df = pd.read_csv(INPUT_PATH)


# ============================================
# CHECK DATA
# ============================================

print("Input file:")
print(INPUT_PATH)

print("\nOriginal shape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

display(df.head())

Input file:
D:\My_Project\Analyse\capston Project_2-Global Supply Chain Risk & Logistics\data\interim\cleaned_data.csv

Original shape:
(5000, 13)

Columns:
['date', 'origin_port', 'destination_port', 'transport_mode', 'product_category', 'distance_km', 'weight_mt', 'fuel_price_index', 'geopolitical_risk_score', 'weather_condition', 'carrier_reliability_score', 'lead_time_days', 'disruption']


,date,origin_port,destination_port,transport_mode,product_category,distance_km,weight_mt,fuel_price_index,geopolitical_risk_score,weather_condition,carrier_reliability_score,lead_time_days,disruption
0,2025-10-16,Singapore,Los Angeles,Rail,Textiles,5930.83,197.42,2.43,5.0,Hurricane,0.865,41.39,1
1,2024-04-24,Singapore,Shanghai,Rail,Automotive,14285.36,237.24,2.30,7.5,Storm,0.592,40.92,1
2,2024-01-26,Rotterdam,Los Angeles,Rail,Perishables,11113.91,427.42,1.78,5.6,Rain,0.673,11.54,0
3,2024-10-08,Busan,Hamburg,Rail,Electronics,9180.55,170.66,3.20,0.8,Hurricane,0.832,53.13,1
4,2024-09-07,Busan,Singapore,Air,Perishables,2762.27,434.96,2.77,1.9,Fog,0.741,0.50,1


In [2]:
def find_column(*names):

    for name in names:

        name = name.lower()

        if name in df.columns:
            return name

    return None

In [3]:
scheduled = find_column(
    "days_for_shipment_scheduled",
    "days_for_shipping_scheduled",
    "scheduled_shipping_days",
    "scheduled_days"
)

actual = find_column(
    "days_for_shipping_real",
    "days_for_shipping",
    "actual_shipping_days",
    "real_shipping_days"
)

created_features = []

if scheduled and actual:

    df["shipping_delay"] = (
        pd.to_numeric(
            df[actual],
            errors="coerce"
        )
        -
        pd.to_numeric(
            df[scheduled],
            errors="coerce"
        )
    )

    df["shipping_delay_abs"] = (
        df["shipping_delay"].abs()
    )

    created_features.extend([
        "shipping_delay",
        "shipping_delay_abs"
    ])

print("Created:", created_features)

Created: []


In [4]:
inventory = find_column(
    "inventory_level",
    "inventory",
    "stock_level"
)

demand = find_column(
    "order_quantity",
    "demand",
    "units_demand",
    "quantity"
)

if inventory and demand:

    inventory_values = pd.to_numeric(
        df[inventory],
        errors="coerce"
    )

    demand_values = pd.to_numeric(
        df[demand],
        errors="coerce"
    )

    df["inventory_demand_ratio"] = (
        inventory_values
        /
        demand_values.replace(0, np.nan)
    )

    created_features.append(
        "inventory_demand_ratio"
    )

In [5]:
profit = find_column(
    "profit_per_order",
    "profit"
)

sales = find_column(
    "sales",
    "sales_per_customer",
    "revenue"
)

if profit and sales:

    profit_values = pd.to_numeric(
        df[profit],
        errors="coerce"
    )

    sales_values = pd.to_numeric(
        df[sales],
        errors="coerce"
    )

    df["profit_margin"] = (
        profit_values
        /
        sales_values.replace(0, np.nan)
    )

    created_features.append(
        "profit_margin"
    )

In [6]:
if sales and demand:

    sales_values = pd.to_numeric(
        df[sales],
        errors="coerce"
    )

    demand_values = pd.to_numeric(
        df[demand],
        errors="coerce"
    )

    df["sales_per_unit"] = (
        sales_values
        /
        demand_values.replace(0, np.nan)
    )

    created_features.append(
        "sales_per_unit"
    )

In [7]:
discount = find_column(
    "order_item_discount_rate",
    "discount_rate",
    "discount"
)

if sales and discount:

    sales_values = pd.to_numeric(
        df[sales],
        errors="coerce"
    )

    discount_values = pd.to_numeric(
        df[discount],
        errors="coerce"
    )

    df["discounted_sales"] = (
        sales_values
        *
        (
            1
            -
            discount_values.fillna(0)
        )
    )

    created_features.append(
        "discounted_sales"
    )

In [8]:
df = df.replace(
    [np.inf, -np.inf],
    np.nan
)

OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Created features:")

for feature in created_features:
    print("-", feature)

print(
    "\nSaved:",
    OUTPUT_PATH.resolve()
)

print(
    "Final shape:",
    df.shape
)

Created features:

Saved: D:\My_Project\Analyse\capston Project_2-Global Supply Chain Risk & Logistics\data\interim\feature_engineered_data.csv
Final shape: (5000, 13)
